# Step 3: Model Training
Train ML models to predict HOMO, LUMO, and Optical Bandgap

In [1]:
# Install required libraries
!pip install pandas numpy scikit-learn tensorflow -q

In [2]:
# Upload files from previous steps
from google.colab import files
print("Upload: Data_Final_merged.xlsx and features_morgan.npy")
uploaded = files.upload()

Upload: Data_Final_merged.xlsx and features_morgan.npy


Saving features_morgan.npy to features_morgan.npy
Saving Data_Final_merged.xlsx to Data_Final_merged.xlsx


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("STEP 3: MODEL TRAINING")
print("="*70)

STEP 3: MODEL TRAINING


In [4]:
# Load data and features
print("\n[1/6] Loading data and features...")
df = pd.read_excel('Data_Final_merged.xlsx')
X = np.load('features_morgan.npy')

y_homo = df['HOMO_A'].values
y_lumo = df['LUMO_A'].values
y_eg = df['EgA_opt'].values

print(f"Features shape: {X.shape}")
print(f"Samples: {len(y_homo)}")


[1/6] Loading data and features...
Features shape: (1571, 2048)
Samples: 1571


In [5]:
# Train/test split
print("\n[2/6] Creating train/test split (80/20)...")
X_train, X_test, y_homo_train, y_homo_test = train_test_split(
    X, y_homo, test_size=0.2, random_state=42
)
_, _, y_lumo_train, y_lumo_test = train_test_split(
    X, y_lumo, test_size=0.2, random_state=42
)
_, _, y_eg_train, y_eg_test = train_test_split(
    X, y_eg, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} molecules")
print(f"Test set: {X_test.shape[0]} molecules")


[2/6] Creating train/test split (80/20)...
Training set: 1256 molecules
Test set: 315 molecules


In [6]:
# Dictionary to store results
results = {}

In [7]:
# [3/6] Train Linear Regression (Baseline)
print("\n[3/6] Training Linear Regression models (baseline)...")
lr_models = {}

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg', y_eg_train, y_eg_test)]:
    lr = LinearRegression()
    lr.fit(X_train, y_tr)
    y_pred = lr.predict(X_test)

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    lr_models[prop] = lr
    results[f'LR_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"  {prop}: R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[3/6] Training Linear Regression models (baseline)...
  HOMO: R²=-0.143, MAE=0.091, RMSE=0.140
  LUMO: R²=-0.275, MAE=0.118, RMSE=0.165
  Eg: R²=-0.045, MAE=0.091, RMSE=0.138


In [8]:
# [4/6] Train Support Vector Regression with hyperparameter tuning
print("\n[4/6] Training SVR models with hyperparameter tuning...")
print("  (This may take 5-10 minutes)")
svr_models = {}

param_grid = {
    'C': [1, 10, 100],
    'gamma': [0.001, 0.01, 0.1],
    'epsilon': [0.01, 0.1]
}

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg', y_eg_train, y_eg_test)]:
    print(f"  Tuning {prop}...")
    svr = SVR(kernel='rbf')
    grid_search = GridSearchCV(svr, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=0)
    grid_search.fit(X_train, y_tr)

    best_svr = grid_search.best_estimator_
    y_pred = best_svr.predict(X_test)

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    svr_models[prop] = best_svr
    results[f'SVR_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"    Best params: {grid_search.best_params_}")
    print(f"    R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[4/6] Training SVR models with hyperparameter tuning...
  (This may take 5-10 minutes)
  Tuning HOMO...
    Best params: {'C': 1, 'epsilon': 0.01, 'gamma': 0.001}
    R²=0.317, MAE=0.077, RMSE=0.108
  Tuning LUMO...
    Best params: {'C': 1, 'epsilon': 0.1, 'gamma': 0.01}
    R²=0.195, MAE=0.103, RMSE=0.131
  Tuning Eg...
    Best params: {'C': 1, 'epsilon': 0.01, 'gamma': 0.001}
    R²=0.357, MAE=0.070, RMSE=0.109


In [9]:
# [5/6] Train Neural Networks (single output)
print("\n[5/6] Training Neural Network models (single output)...")
print("  (This may take 5-10 minutes)")
nn_models = {}

def create_nn_model(input_dim):
    model = keras.Sequential([
        layers.Dense(512, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='mse', metrics=['mae'])
    return model

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg', y_eg_train, y_eg_test)]:
    print(f"  Training {prop}...")
    model = create_nn_model(X_train.shape[1])

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    )

    history = model.fit(
        X_train, y_tr, epochs=100, batch_size=32,
        validation_split=0.2, callbacks=[early_stop], verbose=0
    )

    y_pred = model.predict(X_test, verbose=0).flatten()

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    nn_models[prop] = model
    results[f'NN_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"    R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[5/6] Training Neural Network models (single output)...
  (This may take 5-10 minutes)
  Training HOMO...
    R²=-6.141, MAE=0.271, RMSE=0.350
  Training LUMO...
    R²=-3.159, MAE=0.250, RMSE=0.298
  Training Eg...
    R²=-3.975, MAE=0.268, RMSE=0.302


In [10]:
# [6/6] Train Multi-Output Neural Network
print("\n[6/6] Training Multi-Output Neural Network...")
y_train_multi = np.column_stack([y_homo_train, y_lumo_train, y_eg_train])
y_test_multi = np.column_stack([y_homo_test, y_lumo_test, y_eg_test])

model_multi = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(3)  # 3 outputs: HOMO, LUMO, Eg
])

model_multi.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse', metrics=['mae']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history_multi = model_multi.fit(
    X_train, y_train_multi, epochs=100, batch_size=32,
    validation_split=0.2, callbacks=[early_stop], verbose=0
)

y_pred_multi = model_multi.predict(X_test, verbose=0)

for i, prop in enumerate(['HOMO', 'LUMO', 'Eg']):
    y_pred = y_pred_multi[:, i]
    y_te = y_test_multi[:, i]

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    results[f'NN_Multi_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"  {prop}: R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[6/6] Training Multi-Output Neural Network...
  HOMO: R²=-2.683, MAE=0.189, RMSE=0.251
  LUMO: R²=-1.059, MAE=0.162, RMSE=0.210
  Eg: R²=-1.817, MAE=0.204, RMSE=0.227


In [11]:
# Display results summary
print("\n" + "="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)

comparison_data = []
for model_name, metrics in results.items():
    parts = model_name.split('_')
    model_type = parts[0] if len(parts) == 2 else '_'.join(parts[:2])
    prop = parts[-1]
    comparison_data.append({
        'Model': model_type,
        'Property': prop,
        'R²': metrics['R2'],
        'MAE': metrics['MAE'],
        'RMSE': metrics['RMSE']
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))


MODEL PERFORMANCE SUMMARY
   Model Property        R²      MAE     RMSE
      LR     HOMO -0.142704 0.091227 0.139910
      LR     LUMO -0.275455 0.118028 0.165268
      LR       Eg -0.044938 0.091364 0.138411
     SVR     HOMO  0.316682 0.077337 0.108192
     SVR     LUMO  0.195071 0.102695 0.131291
     SVR       Eg  0.357441 0.070480 0.108538
      NN     HOMO -6.140861 0.271230 0.349749
      NN     LUMO -3.158596 0.250095 0.298421
      NN       Eg -3.974988 0.267544 0.302009
NN_Multi     HOMO -2.683417 0.188853 0.251193
NN_Multi     LUMO -1.059349 0.161713 0.210001
NN_Multi       Eg -1.816838 0.203689 0.227251


In [12]:
# Identify best models
print("\n" + "="*70)
print("BEST MODELS BY PROPERTY")
print("="*70)

for prop in ['HOMO', 'LUMO', 'Eg']:
    prop_results = comparison_df[comparison_df['Property'] == prop]
    best_idx = prop_results['R²'].idxmax()
    best_model = prop_results.loc[best_idx]
    print(f"{prop}: {best_model['Model']} (R²={best_model['R²']:.3f}, MAE={best_model['MAE']:.3f})")


BEST MODELS BY PROPERTY
HOMO: SVR (R²=0.317, MAE=0.077)
LUMO: SVR (R²=0.195, MAE=0.103)
Eg: SVR (R²=0.357, MAE=0.070)


In [13]:
# Save models
print("\n" + "="*70)
print("SAVING MODELS")
print("="*70)

# Save scikit-learn models
with open('models_lr.pkl', 'wb') as f:
    pickle.dump(lr_models, f)
print("✓ Saved: models_lr.pkl")

with open('models_svr.pkl', 'wb') as f:
    pickle.dump(svr_models, f)
print("✓ Saved: models_svr.pkl")

# Save neural network models
for prop, model in nn_models.items():
    model.save(f'model_nn_{prop}.keras')
    print(f"✓ Saved: model_nn_{prop}.keras")

model_multi.save('model_nn_multi.keras')
print("✓ Saved: model_nn_multi.keras")

# Save comparison table
comparison_df.to_csv('model_comparison.csv', index=False)
print("✓ Saved: model_comparison.csv")


SAVING MODELS
✓ Saved: models_lr.pkl
✓ Saved: models_svr.pkl
✓ Saved: model_nn_HOMO.keras
✓ Saved: model_nn_LUMO.keras
✓ Saved: model_nn_Eg.keras
✓ Saved: model_nn_multi.keras
✓ Saved: model_comparison.csv


In [14]:
# Download all model files
from google.colab import files
files.download('models_lr.pkl')
files.download('models_svr.pkl')
files.download('model_nn_multi.keras')
files.download('model_comparison.csv')

print("\n✓ STEP 3 COMPLETE - All models trained and saved")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ STEP 3 COMPLETE - All models trained and saved
